# Análisis exploratorio del corpus del Taller RAG

Solo se usan columnas agregadas: conteos, longitudes, fechas, fuentes e idiomas. **No se lee la columna `text`**
(ni `title` ni `url`), y ninguna celda muestra texto de documentos.

- Preparado: `data/documentos_rag.parquet` (`scripts/prepare_corpus.py --languages es en`).
- Original: `~/.open_corpus_co_es/data/dragons_rag/dragons_rag.parquet`. Solo se leen `document_id`, `created_at` y `source`.
- Cifras de referencia: `docs/diagnostico_corpus.md`.

## 0. Imports, rutas y guarda

In [ ]:
import hashlib, json, math
from collections import Counter
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PREP = ROOT / 'data' / 'documentos_rag.parquet'
ORIG = Path.home() / '.open_corpus_co_es' / 'data' / 'dragons_rag' / 'dragons_rag.parquet'
FIG = ROOT / 'docs' / 'figuras'
OUT_JSON = ROOT / 'docs' / 'analisis_corpus.json'
SHA_REF = 'f4a669f17eaedd357f72b70c2192e685b6d534a94f6ffaeadbced7290b181728'
SOURCES = ['media_cloud', 'bluesky', 'gdelt']

# Únicas columnas que se leen. text, title y url nunca se cargan.
COLS_PREP = ['document_id', 'source', 'lang', 'lang_source', 'word_count', 'created_at_utc',
             'dup_count', 'dup_sources', 'dup_document_ids']
COLS_ORIG = ['document_id', 'created_at', 'source']

schema = pq.read_schema(PREP)
if len(schema) != 41 or 'dup_count' not in schema.names:
    raise RuntimeError(f'{PREP} no es el corpus preparado: tiene {len(schema)} columnas '
                       f"(se esperan 41) y dup_count={'dup_count' in schema.names}. "
                       'Regenéralo con scripts/prepare_corpus.py --languages es en.')
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})


def py(o):
    """Convierte tipos numpy/pandas a tipos JSON."""
    if isinstance(o, dict):
        return {str(k): py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [py(v) for v in o]
    if isinstance(o, (pd.Series, pd.DataFrame)):
        return py(o.to_dict())
    if isinstance(o, np.generic):
        o = o.item()
    if isinstance(o, float) and math.isnan(o):
        return None
    if isinstance(o, pd.Timestamp):
        return o.isoformat()
    return o


R = {}
def reg(key, value, celda, calculo):
    """Registra una cifra para docs/analisis_corpus.json."""
    R[key] = {'valor': py(value), 'celda': celda, 'calculo': calculo}
    return value

## 1. Carga y verificación

In [ ]:
pf_prep, pf_orig = pq.ParquetFile(PREP), pq.ParquetFile(ORIG)
h = hashlib.sha256()
with open(ORIG, 'rb') as f:
    for block in iter(lambda: f.read(1 << 20), b''):
        h.update(block)
sha = h.hexdigest()

reg('preparado_filas', pf_prep.metadata.num_rows, 1, 'metadatos de data/documentos_rag.parquet')
reg('preparado_columnas', len(pf_prep.schema_arrow), 1, 'campos del esquema arrow del preparado')
reg('original_filas', pf_orig.metadata.num_rows, 1, 'metadatos de dragons_rag.parquet')
reg('original_columnas', len(pf_orig.schema_arrow), 1, 'campos del esquema arrow del original')
reg('original_bytes', ORIG.stat().st_size, 1, 'tamaño en disco del original')
reg('original_sha256', sha, 1, 'sha256 del original en bloques de 1 MB')
reg('original_sha256_coincide', sha == SHA_REF, 1, f'comparación con {SHA_REF}')

df = pq.read_table(PREP, columns=COLS_PREP).to_pandas()
N = len(df)
display(pd.DataFrame({
    'filas': [pf_prep.metadata.num_rows, pf_orig.metadata.num_rows],
    'columnas': [len(pf_prep.schema_arrow), len(pf_orig.schema_arrow)],
}, index=['preparado', 'original']))
print('sha256 original:', sha, '| coincide:', sha == SHA_REF)
print('document_id repetidos en el preparado:', reg('preparado_document_id_repetidos',
      int(df.document_id.duplicated().sum()), 1, 'document_id.duplicated().sum()'))

## 2. Composición por fuente e idioma

In [ ]:
# Referencia: docs/diagnostico_corpus.md, sección 12 P2 (escenario --languages es en).
REF = {('media_cloud', 'es'): 59125, ('media_cloud', 'en'): 35176,
       ('bluesky', 'es'): 5160, ('bluesky', 'en'): 46716,
       ('gdelt', 'es'): 381, ('gdelt', 'en'): 365}

by_source = df.source.value_counts()
by_lang = df.lang.value_counts()
ct = pd.crosstab(df.source, df.lang).reindex(SOURCES)
ct_ls = pd.crosstab([df.source, df.lang], df.lang_source)

display(pd.DataFrame({'filas': by_source, '%': (100 * by_source / N).round(2)}))
display(pd.DataFrame({'filas': by_lang, '%': (100 * by_lang / N).round(2)}))
display(ct.assign(total=ct.sum(axis=1), pct_en=(100 * ct['en'] / ct.sum(axis=1)).round(2)))
display(ct_ls)

comp = pd.DataFrame([{'source': s, 'lang': l, 'real': int(ct.loc[s, l]), 'referencia': r,
                      'coincide': int(ct.loc[s, l]) == r} for (s, l), r in REF.items()])
display(comp)

reg('por_fuente', by_source, 2, 'source.value_counts()')
reg('por_fuente_pct', (100 * by_source / N).round(2), 2, '100 * source.value_counts() / N')
reg('por_idioma', by_lang, 2, 'lang.value_counts()')
reg('por_idioma_pct', (100 * by_lang / N).round(2), 2, '100 * lang.value_counts() / N')
reg('fuente_x_idioma', ct.to_dict('index'), 2, 'crosstab(source, lang)')
reg('fuente_idioma_x_origen_idioma', {f'{s}|{l}': row for (s, l), row in ct_ls.to_dict('index').items()},
    2, 'crosstab([source, lang], lang_source)')
reg('comparacion_diagnostico', comp.to_dict('records'), 2, 'crosstab(source, lang) vs. diagnostico_corpus.md §12 P2')
reg('comparacion_diagnostico_todo_coincide', bool(comp.coincide.all()), 2, 'all(coincide)')

## 3. Longitudes (`word_count`)

In [ ]:
bins = np.logspace(0, np.log10(df.word_count.max()) + 0.05, 70)
fig, axes = plt.subplots(len(SOURCES), 1, figsize=(8, 7), sharex=True)
for ax, s in zip(axes, SOURCES):
    w = df.loc[df.source == s, 'word_count']
    ax.hist(w, bins=bins, color='#4c72b0')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.axvline(5000, color='#c44e52', ls='--', lw=1)
    ax.set_title(f'{s} (n = {len(w):,})'.replace(',', '.'), fontsize=10, loc='left')
    ax.set_ylabel('documentos')
axes[-1].set_xlabel('word_count (escala log); línea discontinua = 5.000 palabras')
fig.tight_layout()
fig.savefig(FIG / 'longitudes_por_fuente.png', bbox_inches='tight')
plt.show()

def pct(w):
    return pd.Series({'n': len(w), 'p50': w.quantile(.5), 'p90': w.quantile(.9), 'p95': w.quantile(.95),
                      'p99': w.quantile(.99), 'max': w.max(), 'gt_5000': int((w > 5000).sum()),
                      'gt_5000_pct': round(100 * (w > 5000).mean(), 3)})

lens = pd.concat({s: pct(df.loc[df.source == s, 'word_count']) for s in SOURCES} | {'total': pct(df.word_count)}, axis=1).T
display(lens)
reg('longitud_word_count', lens.to_dict('index'), 3,
    'percentiles 50/90/95/99, máximo y conteo > 5.000 palabras de word_count, por fuente (quantile lineal)')
lens_lang = pd.concat({l: pct(df.loc[df.lang == l, 'word_count']) for l in ['es', 'en']}, axis=1).T
display(lens_lang)
reg('longitud_word_count_por_idioma', lens_lang.to_dict('index'), 3, 'mismas métricas, por lang')

## 4. Distribución temporal (`created_at_utc`)

In [ ]:
year = df.created_at_utc.dt.year
by_year = pd.crosstab(year, df.source).reindex(columns=SOURCES, fill_value=0)
by_year.index = by_year.index.astype(int)
nulls = df.created_at_utc.isna().groupby(df.source).sum().reindex(SOURCES)

fig, ax = plt.subplots(figsize=(9, 4.5))
for s, c in zip(SOURCES, ['#4c72b0', '#55a868', '#c44e52']):
    ax.plot(by_year.index, by_year[s].replace(0, np.nan), marker='o', ms=3, label=s, color=c)
ax.set_yscale('log'); ax.set_xlabel('año (created_at_utc)'); ax.set_ylabel('documentos (escala log)')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))  # años enteros en el eje x
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / 'documentos_por_anio.png', bbox_inches='tight')
plt.show()

display(by_year.assign(total=by_year.sum(axis=1)))
display(nulls.rename('sin_fecha').to_frame())
recent = (year >= 2025)
reg('documentos_por_anio_fuente', {int(y): row for y, row in by_year.to_dict('index').items()}, 4,
    'crosstab(created_at_utc.dt.year, source)')
reg('sin_fecha_por_fuente', nulls, 4, 'created_at_utc.isna() por source')
reg('rango_fechas', {'min': df.created_at_utc.min(), 'max': df.created_at_utc.max()}, 4, 'min/max de created_at_utc')
reg('pct_desde_2025', round(100 * recent.sum() / N, 2), 4, '100 * (año >= 2025) / N, sin fecha cuenta en el denominador')
reg('pct_desde_2025_por_fuente', (100 * recent.groupby(df.source).mean()).round(2), 4, '100 * mean(año >= 2025) por source')

## 5. Duplicados (`dup_count`, `dup_sources`)

In [ ]:
dist = df.dup_count.value_counts().sort_index()
labels = ['1', '2', '3–5', '6–10', '11–50', '51–200', '>200']
bucket = pd.cut(df.dup_count, [0, 1, 2, 5, 10, 50, 200, np.inf], labels=labels).value_counts().reindex(labels)

groups = df[df.dup_count > 1]
n_src = groups.dup_sources.map(lambda x: len(set(x)))
mixed = groups[n_src > 1]
combos = mixed.dup_sources.map(lambda x: ' + '.join(sorted(set(x)))).value_counts()

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(bucket.index.astype(str), bucket.values, color='#4c72b0')
ax.set_yscale('log'); ax.set_xlabel('tamaño de grupo (dup_count)'); ax.set_ylabel('grupos (escala log)')
for x, v in zip(bucket.index.astype(str), bucket.values):
    ax.text(x, v, f'{v:,}'.replace(',', '.'), ha='center', va='bottom', fontsize=8)
fig.tight_layout()
fig.savefig(FIG / 'dup_count_distribucion.png', bbox_inches='tight')
plt.show()

top = groups.nlargest(10, 'dup_count').reset_index(drop=True)
top10 = pd.DataFrame({
    'rango': range(1, len(top) + 1),
    'dup_count': top.dup_count,
    'fuente_conservada': top.source,
    'lang': top.lang,
    'n_fuentes': top.dup_sources.map(lambda x: len(set(x))),
    **{f'copias_{s}': top.dup_sources.map(lambda x, s=s: Counter(x)[s]) for s in SOURCES},
})
display(bucket.rename('grupos').to_frame())
display(combos.rename('grupos').to_frame())
display(top10)

reg('dup_count_distribucion', dist, 5, 'dup_count.value_counts() (1 = sin duplicados)')
reg('dup_count_rangos', bucket, 5, 'pd.cut(dup_count) en rangos')
reg('grupos_con_duplicados', len(groups), 5, 'filas del preparado con dup_count > 1 (una por grupo)')
reg('filas_en_grupos_con_duplicados', int(groups.dup_count.sum()), 5, 'suma de dup_count en esos grupos')
reg('copias_descartadas_en_preparado', int(groups.dup_count.sum() - len(groups)), 5, 'sum(dup_count) - grupos')
reg('dup_count_max', int(df.dup_count.max()), 5, 'max(dup_count)')
reg('grupos_mezclan_fuentes', len(mixed), 5, 'grupos con más de una source distinta en dup_sources')
reg('grupos_mezclan_fuentes_combinaciones', combos, 5, 'combinaciones de fuentes en esos grupos')
reg('top10_grupos', top10.to_dict('records'), 5, 'nlargest(10, dup_count); solo conteos por fuente')

## 6. Propagación de noticia a red (`media_cloud` → `bluesky`)

Grupos del preparado cuyo `dup_sources` incluye `media_cloud` y `bluesky`. Las fechas de cada copia se
resuelven con `document_id` en el original (solo `document_id`, `created_at`, `source`).
`delta_h` = primera aparición en `bluesky` − primera aparición en `media_cloud`, en horas; positivo = Bluesky después.
Una fecha solo con día (`AAAA-MM-DD`) se interpreta como 00:00 UTC, así que su incertidumbre es de ±24 h.

In [ ]:
# Combinaciones de fuentes de todos los grupos con más de una fuente, incluidas las que dan 0.
PAIRS = ['gdelt + media_cloud', 'bluesky + media_cloud', 'bluesky + gdelt', 'bluesky + gdelt + media_cloud']
combo_all = (df.dup_sources.map(lambda x: ' + '.join(sorted(set(x))))
             .loc[lambda s: s.str.contains(' + ', regex=False)].value_counts().reindex(PAIRS, fill_value=0))
display(combo_all.rename('grupos').to_frame())
reg('combinaciones_fuentes_en_grupos', combo_all, 6,
    'grupos del preparado con más de una source en dup_sources, por combinación (incluye combinaciones con 0)')

both = df[df.dup_sources.map(lambda x: 'media_cloud' in x and 'bluesky' in x)]
orig = pq.read_table(ORIG, columns=COLS_ORIG).to_pandas()
is_day = lambda s: s.fillna('').str.fullmatch(r'\d{4}-\d{2}-\d{2}')
res = {
    'grupos_mc_y_bs': len(both), 'n': 0, 'copias_sin_match_en_original': 0,
    'delta_h_p25': None, 'delta_h_mediana': None, 'delta_h_p75': None,
    'delta_dias_p25': None, 'delta_dias_mediana': None, 'delta_dias_p75': None,
    'pct_bluesky_antes': None, 'pct_mc_primera_solo_dia': None, 'pct_bs_primera_solo_dia': None,
    # Resolución real de las fechas en el original; no depende de los grupos.
    'pct_media_cloud_solo_dia_en_todo_el_original': round(100 * is_day(orig.loc[orig.source == 'media_cloud', 'created_at']).mean(), 2),
    'pct_bluesky_solo_dia_en_todo_el_original': round(100 * is_day(orig.loc[orig.source == 'bluesky', 'created_at']).mean(), 2),
}

j = pd.DataFrame()
if len(both):
    ids = (both[['document_id', 'dup_document_ids']].rename(columns={'document_id': 'grupo'})
           .explode('dup_document_ids').rename(columns={'dup_document_ids': 'document_id'}))
    m = ids.merge(orig, on='document_id', how='left', validate='many_to_one')
    m['ts'] = pd.to_datetime(m.created_at, utc=True, errors='coerce', format='mixed')
    m['solo_dia'] = is_day(m.created_at)
    first = (m.dropna(subset=['ts']).sort_values('ts')
              .groupby(['grupo', 'source'])[['ts', 'solo_dia']].first())
    j = (first.xs('media_cloud', level='source')
         .join(first.xs('bluesky', level='source'), lsuffix='_mc', rsuffix='_bs', how='inner'))
    j['delta_h'] = (j.ts_bs - j.ts_mc).dt.total_seconds() / 3600
    j['delta_dias'] = (j.ts_bs.dt.floor('D') - j.ts_mc.dt.floor('D')).dt.days
    q, qd = j.delta_h.quantile([.25, .5, .75]), j.delta_dias.quantile([.25, .5, .75])
    res.update({
        'n': len(j), 'copias_sin_match_en_original': int(m.source.isna().sum()),
        'delta_h_p25': q[.25], 'delta_h_mediana': q[.5], 'delta_h_p75': q[.75],
        'delta_dias_p25': qd[.25], 'delta_dias_mediana': qd[.5], 'delta_dias_p75': qd[.75],
        'pct_bluesky_antes': round(100 * (j.delta_h < 0).mean(), 2),
        'pct_mc_primera_solo_dia': round(100 * j.solo_dia_mc.mean(), 2),
        'pct_bs_primera_solo_dia': round(100 * j.solo_dia_bs.mean(), 2),
    })
else:
    print('Sin grupos con copias en media_cloud y bluesky: n = 0. No se genera la figura de propagación.')
display(pd.Series(res, name='valor').to_frame())

if len(j):
    d = j.delta_h.sort_values().reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.22 * len(d) + 1)))
    ax.scatter(d, d.index, color=np.where(j.sort_values('delta_h').solo_dia_mc, '#dd8452', '#4c72b0'), s=18)
    ax.axvline(0, color='grey', lw=1)
    ax.set_xscale('symlog', linthresh=24)
    ax.set_xlabel('horas de bluesky respecto a media_cloud (symlog; ±24 h lineal)')
    ax.set_ylabel('grupo (ordenado)'); ax.set_yticks([])
    ax.set_title('naranja = fecha de media_cloud solo con día', fontsize=9, loc='left')
    fig.tight_layout()
    fig.savefig(FIG / 'propagacion_mc_bluesky.png', bbox_inches='tight')
    plt.show()

reg('propagacion_mc_bluesky', res, 6,
    'grupos con media_cloud y bluesky en dup_sources; fechas del original por document_id; '
    'delta = min(ts bluesky) - min(ts media_cloud); solo_dia = created_at con formato AAAA-MM-DD')

## 7. Cobertura y cuotas para el gold standard (40 preguntas)

In [ ]:
TOTAL_Q, MIN_EN = 40, 6

def largest_remainder(weights, total):
    """Reparte total en enteros proporcionales a weights (método del resto mayor)."""
    raw = weights / weights.sum() * total
    base = np.floor(raw).astype(int)
    for k in (raw - base).sort_values(ascending=False).index[:total - base.sum()]:
        base[k] += 1
    return base

period = pd.cut(year, [0, 2023, 2024, 2025, 2026], labels=['≤2023', '2024', '2025', '2026']).astype(object).fillna('sin fecha')
prop = {
    'fuente': (100 * df.source.value_counts(normalize=True)).round(2),
    'idioma': (100 * df.lang.value_counts(normalize=True)).round(2),
    'periodo': (100 * period.value_counts(normalize=True)).round(2),
}
for k, v in prop.items():
    display(v.rename(f'% por {k}').to_frame())
prop_year = (100 * year.value_counts(normalize=True, dropna=False).sort_index()).round(3)

def quotas(q_src):
    rows = {s: largest_remainder(ct.loc[s], n) if n else pd.Series(0, index=ct.columns) for s, n in q_src.items()}
    t = pd.DataFrame(rows).T.reindex(SOURCES)[['es', 'en']]
    ajuste = 0
    while t['en'].sum() < MIN_EN:  # mueve preguntas es → en en la fuente con más es
        s = t['es'].idxmax(); t.loc[s, 'es'] -= 1; t.loc[s, 'en'] += 1; ajuste += 1
    return t.assign(total=t.sum(axis=1)), ajuste

q_prop, aj_prop = quotas(largest_remainder(df.source.value_counts(), TOTAL_Q))
q_src_min = largest_remainder(df.source.value_counts(), TOTAL_Q)
for s in SOURCES:  # variante: al menos 1 pregunta por fuente, restada de la fuente mayor
    if q_src_min[s] == 0:
        q_src_min[s] = 1; q_src_min[q_src_min.idxmax()] -= 1
q_min, aj_min = quotas(q_src_min)
q_period = largest_remainder(period[period != 'sin fecha'].value_counts(), TOTAL_Q).reindex(['≤2023', '2024', '2025', '2026'])

print('Cuotas proporcionales (resto mayor), ajuste es→en =', aj_prop); display(q_prop)
print('Variante con mínimo 1 por fuente, ajuste es→en =', aj_min); display(q_min)
print('Referencia por periodo (sin las filas sin fecha)'); display(q_period.rename('preguntas').to_frame())

reg('cobertura_pct', prop, 7, '100 * value_counts(normalize=True) por source, lang y periodo de created_at_utc')
reg('cobertura_pct_por_anio', {('sin fecha' if pd.isna(k) else int(k)): v for k, v in prop_year.items()}, 7,
    '100 * year.value_counts(normalize=True, dropna=False)')
reg('cuotas_40_proporcional', q_prop.to_dict('index'), 7,
    'resto mayor por source y, dentro de cada fuente, por lang; mínimo 6 en inglés')
reg('cuotas_40_proporcional_pct_en', round(100 * q_prop['en'].sum() / TOTAL_Q, 1), 7, '100 * en / 40')
reg('cuotas_40_proporcional_ajuste_en', aj_prop, 7, 'preguntas movidas de es a en para llegar a 6')
reg('cuotas_40_minimo_1_por_fuente', q_min.to_dict('index'), 7, 'igual, con al menos 1 pregunta por fuente')
reg('cuotas_40_por_periodo_referencia', q_period, 7, 'resto mayor por periodo, sin filas sin fecha')

## 8. Exportación

In [ ]:
figs = sorted(p.relative_to(ROOT).as_posix() for p in FIG.glob('*.png'))
reg('figuras', figs, 8, 'PNG guardados en docs/figuras/')
payload = {
    'generado': date.today().isoformat(),
    'notebook': 'notebooks/01_analisis_exploratorio.ipynb',
    'entradas': {'preparado': 'data/documentos_rag.parquet', 'original': str(ORIG).replace(str(Path.home()), '~'),
                 'columnas_leidas_preparado': COLS_PREP, 'columnas_leidas_original': COLS_ORIG},
    'nota': 'Solo cifras agregadas; no contiene texto de documentos.',
    'cifras': R,
}
OUT_JSON.write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'{OUT_JSON.relative_to(ROOT)}: {len(R)} cifras')
print('figuras:', *figs, sep='\n  ')